# Answer one question from two sources

*Data Analysis in Python · Lesson 13 final exercise*

Before Q4 2025's purchases and revenue by device and day go into a report, check them: get them from BigQuery and from the event file, and prove the two agree.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

Add your cells between the login cell and the **Check your work** cell. If you're stuck on a step, open its hint under the **Check your work** cell.

1. In the parameters cell, set `PROJECT_ID` to your own Google Cloud project's ID. Run the parameters cell and the login cell.
2. Write `dry_run_bytes(sql)` from topic 13.2. Write the BigQuery query `bq_sql`: from `TABLE`, the `purchase` events from `BQ_START` to `BQ_END`, grouped by `event_date` and device, with the columns `event_date`, `device`, `purchases` and `revenue` (the sum of `ecommerce.purchase_revenue`, rounded to 2 decimals). Create `bytes_bq`: its dry run's bytes, which should be under `BUDGET`. Then create the `bq_daily` table: the query's result, run with `maximum_bytes_billed=BUDGET`.
3. Create the `duck_daily` table with DuckDB, as in topic 13.3.: the `purchase` events in `EVENTS` from `DUCK_START` to `DUCK_END`, joined to `ORDERS`, grouped by `event_date` and device, with the same four columns. The revenue is the orders' `subtotal`.
4. BigQuery's `event_date` is text, such as `20251128`, and DuckDB's is a date. Turn `bq_daily`'s into a date with `pd.to_datetime()`. Then create the `compared` table: `bq_daily` and `duck_daily` side by side, merged on `event_date` and `device`, one to one, with the suffixes `_bq` and `_duck`.
5. Create the `mismatches` table: the rows of `compared` where the purchases differ, or the revenue differs by more than half a cent.
6. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `bytes_bq` is 135,551,439, under the 200 MB budget. The `bq_daily` and `duck_daily` tables each have 276 rows, with 14,995 purchases and \$1,302,659.72 of revenue. The `compared` table has the same 276 rows, and the `mismatches` table has none. Black Friday's mobile purchases, 270 for \$16,803.75, are Q4's biggest day.

## Parameters

In [ ]:
import duckdb
import pandas as pd
from google.cloud import bigquery
from google.colab import auth

PROJECT_ID = ""  # Your own Google Cloud project's ID: the BigQuery queries run there
TABLE = "simmer-python.simmer_store.events_*"  # Simmer Store's GA4 export, one table per day
BQ_START, BQ_END = "20251001", "20251231"  # Q4 2025, as _TABLE_SUFFIX values
BUDGET = 200_000_000  # 200 MB: the most the BigQuery query may process

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
EVENTS = DATA_URL + "events.parquet"
ORDERS = DATA_URL + "orders.csv"
DUCK_START, DUCK_END = "2025-10-01", "2025-12-31"  # Q4 2025, as dates

In [ ]:
# Log in to Google as yourself. Run it, choose your account and Allow.
auth.authenticate_user()
client = bigquery.Client(project=PROJECT_ID)

In [ ]:
def dry_run_bytes(sql):
    """Return the bytes BigQuery would scan for sql, without running it."""
    config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    return client.query(sql, job_config=config).total_bytes_processed


bq_sql = f"""
SELECT event_date, device.category AS device, COUNT(*) AS purchases, ROUND(SUM(ecommerce.purchase_revenue), 2) AS revenue
FROM `{TABLE}`
WHERE _TABLE_SUFFIX BETWEEN '{BQ_START}' AND '{BQ_END}' AND event_name = 'purchase'
GROUP BY event_date, device
"""
bytes_bq = dry_run_bytes(bq_sql)
bq_daily = client.query(bq_sql, job_config=bigquery.QueryJobConfig(maximum_bytes_billed=BUDGET)).to_dataframe()
bytes_bq, len(bq_daily)

In [ ]:
duck_sql = f"""
SELECT e.event_date, e.device_category AS device, COUNT(*) AS purchases, ROUND(SUM(o.subtotal), 2) AS revenue
FROM '{EVENTS}' AS e
JOIN '{ORDERS}' AS o ON e.transaction_id = o.order_id
WHERE e.event_name = 'purchase' AND e.event_date BETWEEN '{DUCK_START}' AND '{DUCK_END}'
GROUP BY e.event_date, device
"""
duck_daily = duckdb.sql(duck_sql).df()
len(duck_daily)

In [ ]:
bq_daily["event_date"] = pd.to_datetime(bq_daily["event_date"], format="%Y%m%d")
compared = bq_daily.merge(duck_daily, on=["event_date", "device"], suffixes=("_bq", "_duck"), validate="one_to_one")
different_purchases = compared["purchases_bq"] != compared["purchases_duck"]
different_revenue = (compared["revenue_bq"] - compared["revenue_duck"]).abs() > 0.005
mismatches = compared[different_purchases | different_revenue]
len(compared), len(mismatches)

In [ ]:
compared.sort_values("revenue_bq", ascending=False).head()

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
    message = f"{type(error).__name__}: {error}" if error is not None else ""
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    elif checked_before:
        results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
    elif [source for source in history if source.strip() and history.count(source) > 1]:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        first_line = repeated[0].strip().splitlines()[0]
        results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
    elif "bytes billed" in message:
        results.append((None, "BigQuery stopped the query over the budget, as maximum_bytes_billed should. Bring bq_sql's scan under BUDGET, then choose Runtime → Restart session and run all."))
    elif "trying to merge on" in message:
        results.append((None, f"The merge stopped: {error}. Turn bq_daily's event_date into a date with pd.to_datetime(..., format=\"%Y%m%d\") before you merge."))
    elif "bigquery.jobs.create" in message:
        results.append((None, "BigQuery answered 403: your queries can't run in this project. Set PROJECT_ID to your own project's ID, not simmer-python, then choose Runtime → Restart session and run all."))
    elif error is not None:
        results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
    else:
        results.append((True, "Every cell in this session ran once, with no errors."))

    budget = found.get("BUDGET", 200_000_000)
    bytes_bq = found.get("bytes_bq")
    if not pd.api.types.is_number(bytes_bq):
        results.append((False, "bytes_bq should be a number: the dry run's total_bytes_processed for bq_sql."))
    elif bytes_bq > budget and bytes_bq > 600_000_000 and bytes_bq < 1_000_000_000:
        results.append((False, f"bytes_bq is {bytes_bq:,}, over the budget of {budget:,}. The query reads every day's table: pick Q4's tables with _TABLE_SUFFIX instead of event_date."))
    elif bytes_bq > budget:
        results.append((False, f"bytes_bq is {bytes_bq:,}, over the budget of {budget:,}. Name only the columns you need, and pick Q4's tables with _TABLE_SUFFIX."))
    else:
        results.append((True, f"The BigQuery query processes {bytes_bq:,} bytes, under the budget of {budget:,}."))

    ran_with_limit = isinstance(inputs, list) and any("maximum_bytes_billed" in source for source in inputs[1:-1])
    columns = ["event_date", "device", "purchases", "revenue"]
    totals = {}
    for name, revenue_name in [("bq_daily", "ecommerce.purchase_revenue"), ("duck_daily", "the orders' subtotal")]:
        table = found.get(name)
        if not isinstance(table, pd.DataFrame) or [column for column in columns if column not in table.columns]:
            results.append((False, f"{name} should be a DataFrame with the columns {', '.join(columns)}."))
            continue
        purchases, revenue = int(table["purchases"].sum()), float(table["revenue"].sum())
        totals[name] = (len(table), purchases, revenue)
        if purchases > 100000:
            results.append((False, f"{name} counts {purchases:,} events, not purchases. Keep only the purchases: event_name = 'purchase'."))
        elif len(table) != 276:
            results.append((False, f"{name} should have 276 rows: 92 days times 3 devices. It has {len(table)}. Check the dates and GROUP BY event_date and device."))
        elif purchases != 14995:
            results.append((False, f"Q4 2025 has 14,995 purchases. {name} has {purchases:,}. Check the dates."))
        elif name == "duck_daily" and abs(revenue - 1450951.25) < 0.05:
            results.append((False, "duck_daily's revenue is $1,450,951.25: the orders' total, with shipping and tax. GA4's purchase revenue is the subtotal: SUM(o.subtotal)."))
        elif abs(revenue - 1302659.72) > 0.05:
            results.append((False, f"Q4 2025's purchase revenue is $1,302,659.72. {name} has ${revenue:,.2f}. Sum {revenue_name}."))
        elif name == "bq_daily" and not ran_with_limit:
            results.append((False, "Run bq_sql with a limit: bigquery.QueryJobConfig(maximum_bytes_billed=BUDGET)."))
        else:
            results.append((True, f"{name} has 276 rows, with 14,995 purchases and $1,302,659.72 of revenue."))

    compared = found.get("compared")
    mismatches = found.get("mismatches")
    needed = ["event_date", "device", "purchases_bq", "purchases_duck", "revenue_bq", "revenue_duck"]
    if not isinstance(compared, pd.DataFrame) or [column for column in needed if column not in compared.columns]:
        results.append((False, f"compared should be a DataFrame with the columns {', '.join(needed)}: bq_daily.merge(duck_daily, on=[\"event_date\", \"device\"], suffixes=(\"_bq\", \"_duck\"))."))
    elif len(compared) != 276:
        results.append((False, f"compared should have the 276 rows both tables share. It has {len(compared)}. Merge on event_date and device, with dates on both sides."))
    elif not isinstance(mismatches, pd.DataFrame):
        results.append((False, "mismatches should be a DataFrame: the rows of compared where the purchases or the revenue differ."))
    elif len(mismatches) > 0:
        results.append((False, f"mismatches has {len(mismatches)} rows. If both tables have the right totals, check the conditions: the purchases differ, or the revenue differs by more than 0.005."))
    else:
        results.append((True, "BigQuery and DuckDB agree on all 276 days and devices, to the cent."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

## Hints

Stuck on a step? Open its hint. Each one says where the course showed it, and gives the shape of the code, with `...` where your part goes. Try the step from the hint before you open the solution.

<details>
<summary><b>Step 2: BigQuery, under the budget</b></summary>

Topic 13.2.: dry-run the query first, and pick the tables with `_TABLE_SUFFIX`, not with `event_date`, which would read every day's table.

~~~python
def dry_run_bytes(sql):
    config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    return client.query(sql, job_config=config).total_bytes_processed


bq_sql = f"""
SELECT event_date, device.category AS device, COUNT(*) AS purchases, ROUND(SUM(...), 2) AS revenue
FROM `{TABLE}`
WHERE _TABLE_SUFFIX BETWEEN '{BQ_START}' AND '{BQ_END}' AND ...
GROUP BY event_date, device
"""
bytes_bq = dry_run_bytes(bq_sql)
bq_daily = client.query(bq_sql, job_config=bigquery.QueryJobConfig(maximum_bytes_billed=...)).to_dataframe()
~~~

</details>

<details>
<summary><b>Step 3: DuckDB, with the orders</b></summary>

Topic 13.3.: join the events to the orders with aliases, and sum the subtotal.

~~~python
duck_sql = f"""
SELECT e.event_date, e.device_category AS device, COUNT(*) AS purchases, ROUND(SUM(...), 2) AS revenue
FROM '{EVENTS}' AS e
JOIN '{ORDERS}' AS o ON ...
WHERE e.event_name = 'purchase' AND e.event_date BETWEEN '{DUCK_START}' AND '{DUCK_END}'
GROUP BY e.event_date, device
"""
duck_daily = duckdb.sql(duck_sql).df()
~~~

</details>

<details>
<summary><b>Step 4: side by side</b></summary>

Lesson 3.'s `pd.to_datetime()` with a `format=`, and lesson 5.'s `merge()` with `validate=`.

~~~python
bq_daily["event_date"] = pd.to_datetime(bq_daily["event_date"], format=...)
compared = bq_daily.merge(duck_daily, on=[...], suffixes=("_bq", "_duck"), validate=...)
~~~

</details>

<details>
<summary><b>Step 5: the mismatches</b></summary>

A boolean filter with two conditions, joined with `|`, as in lesson 2.

~~~python
different_purchases = compared["purchases_bq"] != ...
different_revenue = (compared["revenue_bq"] - ...).abs() > 0.005
mismatches = compared[different_purchases | different_revenue]
~~~

</details>

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
def dry_run_bytes(sql):
    """Return the bytes BigQuery would scan for sql, without running it."""
    config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    return client.query(sql, job_config=config).total_bytes_processed


bq_sql = f"""
SELECT event_date, device.category AS device, COUNT(*) AS purchases, ROUND(SUM(ecommerce.purchase_revenue), 2) AS revenue
FROM `{TABLE}`
WHERE _TABLE_SUFFIX BETWEEN '{BQ_START}' AND '{BQ_END}' AND event_name = 'purchase'
GROUP BY event_date, device
"""
bytes_bq = dry_run_bytes(bq_sql)
bq_daily = client.query(bq_sql, job_config=bigquery.QueryJobConfig(maximum_bytes_billed=BUDGET)).to_dataframe()
bytes_bq, len(bq_daily)
```

```python
duck_sql = f"""
SELECT e.event_date, e.device_category AS device, COUNT(*) AS purchases, ROUND(SUM(o.subtotal), 2) AS revenue
FROM '{EVENTS}' AS e
JOIN '{ORDERS}' AS o ON e.transaction_id = o.order_id
WHERE e.event_name = 'purchase' AND e.event_date BETWEEN '{DUCK_START}' AND '{DUCK_END}'
GROUP BY e.event_date, device
"""
duck_daily = duckdb.sql(duck_sql).df()
len(duck_daily)
```

```python
bq_daily["event_date"] = pd.to_datetime(bq_daily["event_date"], format="%Y%m%d")
compared = bq_daily.merge(duck_daily, on=["event_date", "device"], suffixes=("_bq", "_duck"), validate="one_to_one")
different_purchases = compared["purchases_bq"] != compared["purchases_duck"]
different_revenue = (compared["revenue_bq"] - compared["revenue_duck"]).abs() > 0.005
mismatches = compared[different_purchases | different_revenue]
len(compared), len(mismatches)
```

```python
compared.sort_values("revenue_bq", ascending=False).head()
```

</details>